In [ ]:
import pandas as pd
import numpy as np

# Importing dataset and naming it as df
df = pd.read_csv("/content/seer colon cancer capstone 14-7-2026.txt")


In [ ]:
# checking df size and number of rows and columns
df.info()


In [ ]:
# Print all column names in the DataFrame
print("Variables in the dataset:")
for col in df.columns:
    print(col)


In [ ]:
# Keep only the variables the analysis needs and drop all the other variables
KEEP = {

    # Outcome 1 — late stage
    "Combined Summary Stage with Expanded Regional Codes (2004+)":  "summary_stage",

    # Outcome 2 — surgery
    "Reason no cancer-directed surgery":                            "reason_no_surgery",

    # predictors, all four RQs (RQ1, RQ2, RQ3, RQ4)
    "Age recode with <1 year olds and 90+":                         "age_recode",
    "Age recode with single ages and 90+":                          "age_single",
    "Sex":                                                          "sex",
    "Race and origin recode (NHW, NHB, NHAIAN, NHAPI, Hispanic)":   "race_eth",
    "Primary Site - labeled":                                       "primary_site",
    "Median household income inflation adj to 2024":                "county_income",
    "Rural-Urban Continuum Code":                                   "rucc",
    "Marital status at diagnosis":                                  "marital",

    # predictors for surgery models only (RQ3, RQ4)
    "Derived Summary Grade 2018 (2018+)":                           "grade",
    "Histologic Type ICD-O-3":                                      "histology",
    "Tumor Size Summary (2016+)":                                   "tumor_size",
}

df = df[list(KEEP)].rename(columns=KEEP)

df.info()

In [ ]:
# Print all column names in the DataFrame after choosing the needed variables
print("Variables in the dataset:")
for col in df.columns:
    print(col)

In [ ]:
#checking levels for summary_stage
df['summary_stage'].value_counts(dropna=False)


In [ ]:
# Stage: rename levels to localized, regional, distant and np.nan
stage_map = {
    "In situ":                                                      "Localized",
    "Localized only":                                               "Localized",
    "Regional by direct extension only":                            "Regional",
    "Regional lymph nodes involved only":                           "Regional",
    "Regional by both direct extension and lymph node involvement": "Regional",
    "Distant site(s)/node(s) involved":                             "Distant",
    "Unknown/unstaged/unspecified/DCO":                               np.nan,
}

df["stage"] = df["summary_stage"].map(stage_map)

# Outcome for RQ1/RQ2 which is late stage pressentation
df["late_stage"] = df["stage"].map({"Localized": 0, "Regional": 1, "Distant": 1})

print(df["stage"].value_counts(dropna=False))
print(df["late_stage"].value_counts(dropna=False, normalize=True))

In [ ]:
#checking levels for reason_no_surgery

df['reason_no_surgery'].value_counts(dropna=False)


In [ ]:
# Surgery: did the patient receive cancer-directed surgery? Binary outcome, relevelling to 1 (recieved surgery) and 0 (Not recieved surgery) and np.nan
surgery_map = {
    "Surgery performed":                                                            1,
    "Not recommended":                                                              0,
    "Not recommended, contraindicated due to other cond; autopsy only (1973-2002)": 0,
    "Recommended but not performed, patient refused":                               0,
    "Recommended but not performed, unknown reason":                                0,
    "Not performed, patient died prior to recommended surgery":                     0,
    "Recommended, unknown if performed":                                            np.nan,
    "Unknown; death certificate; or autopsy only (2003+)":                          np.nan,
}

df["surgery"] = df["reason_no_surgery"].map(surgery_map)

print(df["surgery"].value_counts(dropna=False))

In [ ]:
#checking levels for age_recode
df['age_recode'].value_counts(dropna=False)


In [ ]:
# Relevelling age to become 4 bands
age_map = {
    "15-19 years": "<50",   "20-24 years": "<50",   "25-29 years": "<50",
    "30-34 years": "<50",   "35-39 years": "<50",   "40-44 years": "<50",
    "45-49 years": "<50",
    "50-54 years": "50-64", "55-59 years": "50-64", "60-64 years": "50-64",
    "65-69 years": "65-74", "70-74 years": "65-74",
    "75-79 years": "75+",   "80-84 years": "75+",   "85-89 years": "75+",
    "90+ years":   "75+",
}

df["age_band"] = df["age_recode"].map(age_map)

df["age_band"] = pd.Categorical(
    df["age_band"],
    categories=["50-64", "<50", "65-74", "75+"],
    ordered=False,
)

print(df["age_band"].value_counts())

In [ ]:
# cleaning numerical age
df["age_num"] = (df["age_single"]
                 .str.extract(r"(\d+)")[0]
                 .astype(int))

In [ ]:
#checking age_num

df['age_num'].value_counts(dropna=False)


In [ ]:
#checking sex

df['sex'].value_counts(dropna=False)


In [ ]:
#checking race_eth

df['race_eth'].value_counts(dropna=False)


In [ ]:
#arranging categories for race where white is the reference
df["race_eth"] = pd.Categorical(df["race_eth"], categories=[
    "Non-Hispanic White", "Non-Hispanic Black", "Hispanic (All Races)",
    "Non-Hispanic Asian or Pacific Islander",
    "Non-Hispanic American Indian/Alaska Native", "Non-Hispanic Unknown Race"])

In [ ]:
#checking primary_site

df['primary_site'].value_counts(dropna=False)


In [ ]:
# recoding site into proximal (right) vs distal (left) colon
site_map = {
    "C18.0-Cecum":                    "Proximal",
    "C18.2-Ascending colon":          "Proximal",
    "C18.3-Hepatic flexure of colon": "Proximal",
    "C18.4-Transverse colon":         "Proximal",
    "C18.5-Splenic flexure of colon": "Distal",
    "C18.6-Descending colon":         "Distal",
    "C18.7-Sigmoid colon":            "Distal",
}

df["subsite"] = df["primary_site"].map(site_map)

print(df["subsite"].value_counts())

In [ ]:
#checking acounty_income

df['county_income'].value_counts(dropna=False)


In [ ]:
# County median household income, cleaned into 5 ordered bands
income_map = {
    "< $40,000":                              "< $60k",
    "$40,000 - $44,999":                      "< $60k",
    "$45,000 - $49,999":                      "< $60k",
    "$50,000 - $54,999":                      "< $60k",
    "$55,000 - $59,999":                      "< $60k",
    "$60,000 - $64,999":                      "$60-75k",
    "$65,000 - $69,999":                      "$60-75k",
    "$70,000 - $74,999":                      "$60-75k",
    "$75,000 - $79,999":                      "$75-90k",
    "$80,000 - $84,999":                      "$75-90k",
    "$85,000 - $89,999":                      "$75-90k",
    "$90,000 - $94,999":                      "$90-110k",
    "$95,000 - $99,999":                      "$90-110k",
    "$100,000 - $109,999":                    "$90-110k",
    "$110,000 - $119,999":                    "$110k+",
    "$120,000+":                              "$110k+",
    "Unknown/missing/no match/Not 1990-2024": np.nan,
}

df["income5"] = df["county_income"].map(income_map)

df["income5"] = pd.Categorical(
    df["income5"],
    categories=["< $60k", "$60-75k", "$75-90k", "$90-110k", "$110k+"],
    ordered=True,
)

print(df["income5"].value_counts(dropna=False).sort_index())

In [ ]:
#checking rucc

df['rucc'].value_counts(dropna=False)


In [ ]:
# Rurality: recoding into metro vs non-metro and np.nan
rucc_map = {
    "Counties in metropolitan areas ge 1 million pop":              "Metro",
    "Counties in metropolitan areas of 250,000 to 1 million pop":   "Metro",
    "Counties in metropolitan areas of lt 250 thousand pop":        "Metro",
    "Nonmetropolitan counties adjacent to a metropolitan area":     "NonMetro",
    "Nonmetropolitan counties not adjacent to a metropolitan area": "NonMetro",
    "Unknown/missing/no match (Alaska or Hawaii - Entire State)":   np.nan,
    "Unknown/missing/no match/Not 1990-2024":                       np.nan,
}

df["rurality"] = df["rucc"].map(rucc_map)

df["rurality"] = pd.Categorical(df["rurality"], categories=["Metro", "NonMetro"])

print(df["rurality"].value_counts(dropna=False))

In [ ]:
#checking marital

df['marital'].value_counts(dropna=False)


In [ ]:
# Marital status: recoding into married vs unmarried and Unknown
marital_map = {
    "Married (including common law)": "Married",
    "Single (never married)":         "Unmarried",
    "Widowed":                        "Unmarried",
    "Divorced":                       "Unmarried",
    "Separated":                      "Unmarried",
    "Unmarried or Domestic Partner":  "Unmarried",
    "Unknown":                        "Unknown",
}

df["marital"] = df["marital"].map(marital_map)

df["marital"] = pd.Categorical(df["marital"], categories=["Married", "Unmarried", "Unknown"])

print(df["marital"].value_counts(dropna=False))

In [ ]:
#checking grade

df['grade'].value_counts(dropna=False)


In [ ]:
# Grade: recoding into 3-levels low vs high vs Unknown
grade_map = {
    "Site-specific grade system category (1)": "Low",      # well differentiated
    "Site-specific grade system category (2)": "Low",      # moderately differentiated
    "Site-specific grade system category (3)": "High",     # poorly differentiated
    "Site-specific grade system category (4)": "High",     # undifferentiated
    "Grade cannot be assessed; Unknown":       "Unknown", #Unknown
}

df["grade"] = df["grade"].map(grade_map)

df["grade"] = pd.Categorical(df["grade"], categories=["Low", "High", "Unknown"])
print(df["grade"].value_counts())

In [ ]:
#checking histology

df['histology'].value_counts(dropna=False)


In [ ]:
# Histology: recoding into 2 levels — conventional adenocarcinoma vs variant
hist_map = {
    8480: "Variant",   # mucinous
    8481: "Variant",   # mucinous
    8490: "Variant",   # signet-ring
}

# everything else in the cohort is conventional adenocarcinoma
df["histology"] = df["histology"].map(hist_map).fillna("AdenoNOS")

df["histology"] = pd.Categorical(
    df["histology"],
    categories=["AdenoNOS", "Variant"],
    ordered=False,
)

print(df["histology"].value_counts())

In [ ]:
#checking tumor size

df['tumor_size'].value_counts(dropna=False)


In [ ]:
import numpy as np

df["tumor_size"] = pd.to_numeric(df["tumor_size"], errors="coerce")

# SEER special codes that need to become np.nan
#   000 = no mass found            989      = >=989mm
#   990 = microscopic focus        991-998  = descriptive ("<2cm", "diffuse")
#   999 = unknown
df.loc[(df["tumor_size"] == 0) | (df["tumor_size"] >= 989), "tumor_size"] = np.nan

print(df["tumor_size"].describe())

In [ ]:
# Band tumor size. recoding tumor size into 4 levels
df["size_band"] = pd.cut(df["tumor_size"],
                         bins=[0, 20, 40, 60, 250],
                         labels=["<=20mm", "21-40mm", "41-60mm", ">60mm"])

df["size_band"] = df["size_band"].cat.add_categories("Unknown").fillna("Unknown")

print(df["size_band"].value_counts())

In [ ]:
# Print all column names in the DataFrame
print("Variables in the dataset:")
for col in df.columns:
    print(col)


In [ ]:
# Drop the raw source columns because   the cleaned versions exist
df = df.drop(columns=[
    "summary_stage",       # -> stage, late_stage
    "reason_no_surgery",   # -> surgery
    "primary_site",        # -> subsite
    "age_recode",          # -> age_band
    "age_single",          # -> age_num
    "rucc",                # -> rurality
    "county_income",       # -> income5
    "tumor_size",       # -> size_band
])


In [ ]:
df.info()

In [ ]:
# Chekcing dupliate rows. However, SEER is de-identified with no patient ID, so duplicated() flags patients who share every characteristic,  and are not real duplicate records.
# So there is no need to drop any duplicates as they are not real duplicates
df.duplicated().sum()

In [ ]:
# Checking Missingness for every column
missing = df.isna().sum().to_frame("missing_n")
missing["missing_pct"] = (missing["missing_n"] / len(df) * 100).round(2)

missing.sort_values("missing_n")

In [ ]:
# Dropping missing values from df from these values: income5, rurality, stage, late_stage and surgery.
print("before dropping:", len(df))

df = df.dropna()

print("after dropping: ", len(df))

In [ ]:
#checking dataframe composition
cols = ["sex", "age_band", "race_eth", "subsite", "income5", "rurality",
        "marital", "grade", "histology", "size_band", "stage", "surgery"]

comp = pd.concat([df[c].value_counts() for c in cols], keys=cols).to_frame("n")
comp["pct"] = (comp["n"] / len(df) * 100).round(2)

comp

In [ ]:
# Describing age_num
df["age_num"].describe().round(1)

In [ ]:
# RQ1 and RQ2 use the whole clean dataset.
# RQ3 and RQ4 use non-metastatic patients only. The event modelled is the absence of surgery, so will flip the coding of surgery so that 1 means no surgery and 0 means surgery
surg_df = df[df["stage"].isin(["Localized", "Regional"])].copy()
surg_df["no_surgery"] = 1 - surg_df["surgery"]

print("RQ1 / RQ2 sample ", len(df))
print("late-stage rate  ", round(df["late_stage"].mean(), 4))
print("RQ3 / RQ4 sample ", len(surg_df))
print("no-surgery rate  ", round(surg_df["no_surgery"].mean(), 4))

In [ ]:
import matplotlib.pyplot as plt

#  patient characteristics.
fig, axes = plt.subplots(2, 3, figsize=(15, 8))

for var, ax in zip(["sex", "age_band", "race_eth", "marital", "income5", "rurality"], axes.flat):
    pct = df[var].value_counts(normalize=True).sort_index().mul(100)
    ax.barh(pct.index.astype(str), pct, color="steelblue")
    ax.set_title(var)
    ax.set_xlabel("% of patients")
    ax.invert_yaxis()

plt.tight_layout()
plt.show()

In [ ]:
#  tumor characteristics and the two outcomes
fig, axes = plt.subplots(2, 3, figsize=(15, 8))

surgery_labels = df["surgery"].map({1.0: "Performed", 0.0: "Not performed"})

for var, ax in zip(["subsite", "grade", "histology", "size_band", "stage"], axes.flat):
    pct = df[var].value_counts(normalize=True).sort_index().mul(100)
    ax.barh(pct.index.astype(str), pct, color="seagreen")
    ax.set_title(var)
    ax.set_xlabel("% of patients")
    ax.invert_yaxis()

pct = surgery_labels.value_counts(normalize=True).mul(100)
axes.flat[5].barh(pct.index.astype(str), pct, color="seagreen")
axes.flat[5].set_title("surgery")
axes.flat[5].set_xlabel("% of patients")
axes.flat[5].invert_yaxis()

plt.tight_layout()
plt.show()

In [ ]:
#  stage by age band.
pd.crosstab(df["age_band"], df["stage"], normalize="index").mul(100).plot(
    kind="bar", stacked=True, figsize=(7, 4))

plt.ylabel("% of patients")
plt.title("Stage distribution by age band")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
#  late-stage rate by the four equity variables.
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

for var, ax in zip(["race_eth", "income5", "rurality", "marital"], axes.flat):
    rate = df.groupby(var, observed=True)["late_stage"].mean().mul(100)
    ax.barh(rate.index.astype(str), rate, color="steelblue")
    ax.axvline(df["late_stage"].mean() * 100, color="red", linestyle="--")
    ax.set_xlim(rate.min() - 1, rate.max() + 1)
    ax.set_title(var)
    ax.set_xlabel("% late stage")
    ax.invert_yaxis()

plt.tight_layout()
plt.show()

for var in ["race_eth", "income5", "rurality", "marital"]:
    print(df.groupby(var, observed=True)["late_stage"].mean().mul(100).round(2), "\n")

In [ ]:
#  late-stage rate by subsite and sex
late_by_site = df.groupby(["subsite", "sex"], observed=True)["late_stage"].mean().mul(100).unstack()

ax = late_by_site.plot(kind="bar", figsize=(7, 4))
ax.axhline(df["late_stage"].mean() * 100, color="red", linestyle="--")
ax.set_ylim(late_by_site.min().min() - 1, late_by_site.max().max() + 1)

plt.ylabel("% late stage")
plt.title("Late-stage rate by subsite and sex")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

print(late_by_site.round(2))

In [ ]:
#  no-surgery rate among non-metastatic patients.
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for var, ax in zip(["age_band", "income5", "rurality"], axes.flat):
    rate = surg_df.groupby(var, observed=True)["no_surgery"].mean().mul(100)
    ax.bar(rate.index.astype(str), rate, color="darkorange")
    ax.axhline(surg_df["no_surgery"].mean() * 100, color="red", linestyle="--")
    ax.set_ylim(max(0, rate.min() - 1), rate.max() + 1)
    ax.set_title(var)
    ax.set_ylabel("% with no surgery")
    ax.tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

for var in ["age_band", "income5", "rurality"]:
    print(surg_df.groupby(var, observed=True)["no_surgery"].mean().mul(100).round(2), "\n")

In [ ]:
# tumor size band by surgery status
size_by_surgery = pd.crosstab(surg_df["surgery"], surg_df["size_band"], normalize="index").mul(100)
size_by_surgery.index = ["No surgery", "Surgery performed"]

size_by_surgery.plot(kind="bar", figsize=(8, 4))

plt.ylabel("% of patients")
plt.title("Tumor size band by surgery status")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

print(size_by_surgery.round(1))

In [ ]:
# Chi square tests for late stage
from scipy.stats import chi2_contingency

stage_preds = ["age_band", "sex", "race_eth", "subsite", "income5", "rurality", "marital"]
surg_preds  = stage_preds + ["grade", "histology", "size_band"]

# Late stage
for var in stage_preds:
    chi2, p, dof, expected = chi2_contingency(pd.crosstab(df[var], df["late_stage"]))
    print(f"{var:10s}  chi2 = {chi2:8.1f}   df = {dof}   p = {p:.3g}")

In [ ]:
# Chi square tests for no surgery
for var in surg_preds:
    chi2, p, dof, expected = chi2_contingency(pd.crosstab(surg_df[var], surg_df["no_surgery"]))
    print(f"{var:10s}  chi2 = {chi2:8.1f}   df = {dof}   p = {p:.3g}")

In [ ]:
# One-hot encoding, dropping the first level of each predictor as the reference to be able to calculate adjusted odds ratios for  RQ1 and RQ3
X_stage = pd.get_dummies(df[stage_preds], drop_first=True).astype(float)
X_surg  = pd.get_dummies(surg_df[surg_preds], drop_first=True).astype(float)

X_stage.shape[1], X_surg.shape[1]

In [ ]:
import statsmodels.api as sm

# RQ1 - which characteristics go with a late-stage diagnosis
rq1 = sm.Logit(df["late_stage"], sm.add_constant(X_stage)).fit()

or_rq1 = pd.DataFrame({
    "OR":      np.exp(rq1.params),
    "CI low":  np.exp(rq1.conf_int()[0]),
    "CI high": np.exp(rq1.conf_int()[1]),
    "p":       rq1.pvalues,
}).round(3)

or_rq1

In [ ]:
# RQ3 — which characteristics go with not receiving surgery.
# The event is surgery = 0, so an OR above 1 means higher odds of not recieving surgery
rq3 = sm.Logit(surg_df["no_surgery"], sm.add_constant(X_surg)).fit()

or_rq3 = pd.DataFrame({
    "OR":      np.exp(rq3.params),
    "CI low":  np.exp(rq3.conf_int()[0]),
    "CI high": np.exp(rq3.conf_int()[1]),
    "p":       rq3.pvalues,
}).round(3)

or_rq3

In [ ]:
#RQ2, predicting late stage
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (roc_auc_score, accuracy_score, precision_score,
                             recall_score, f1_score, brier_score_loss, confusion_matrix)

y = df["late_stage"]

# Stratified so train and test carry the same late-stage rate
X_train, X_test, y_train, y_test = train_test_split(
    X_stage, y, test_size=0.2, stratify=y, random_state=42)

log2 = LogisticRegression(max_iter=1000).fit(X_train, y_train)
rf2  = RandomForestClassifier(n_estimators=300, min_samples_leaf=20,
                              random_state=42, n_jobs=-1).fit(X_train, y_train)

log2_prob, log2_pred = log2.predict_proba(X_test)[:, 1], log2.predict(X_test)
rf2_prob,  rf2_pred  = rf2.predict_proba(X_test)[:, 1],  rf2.predict(X_test)

len(X_train), len(X_test)

In [ ]:
rq2_results = pd.DataFrame({
    "Logistic regression": [
        cross_val_score(log2, X_train, y_train, cv=5, scoring="roc_auc").mean(),
        roc_auc_score(y_test, log2_prob),
        accuracy_score(y_test, log2_pred),
        precision_score(y_test, log2_pred),
        recall_score(y_test, log2_pred),
        f1_score(y_test, log2_pred)],
    "Random forest": [
        cross_val_score(rf2, X_train, y_train, cv=5, scoring="roc_auc").mean(),
        roc_auc_score(y_test, rf2_prob),
        accuracy_score(y_test, rf2_pred),
        precision_score(y_test, rf2_pred),
        recall_score(y_test, rf2_pred),
        f1_score(y_test, rf2_pred)],
}, index=["CV AUROC", "AUROC", "Accuracy", "Precision", "Recall", "F1"]).round(3)

rq2_results

In [ ]:
#Confusion matrix
print(confusion_matrix(y_test, log2_pred))
print(confusion_matrix(y_test, rf2_pred))

In [ ]:
from sklearn.inspection import permutation_importance

# Permutation importance
perm2 = permutation_importance(rf2, X_test, y_test, n_repeats=5,
                               scoring="roc_auc", random_state=42, n_jobs=-1)

imp2 = pd.DataFrame({
    "RF permutation":  perm2.importances_mean,
    "RF impurity":     rf2.feature_importances_,
    "Logistic |coef|": np.abs(log2.coef_[0]),
}, index=X_train.columns).sort_values("RF permutation", ascending=False).round(4)

imp2

In [ ]:
# Permutation importance figure

imp2["RF permutation"].sort_values().plot(kind="barh", figsize=(8, 7), color="steelblue")

plt.xlabel("Drop in AUROC when the feature is shuffled")
plt.title("RQ2 feature importance")
plt.tight_layout()
plt.show()

In [ ]:
# RQ4, predicting no surgery
# the classes are weighted because the outcome is rare
y4 = surg_df["no_surgery"]

X4_train, X4_test, y4_train, y4_test = train_test_split(
    X_surg, y4, test_size=0.2, stratify=y4, random_state=42)

log4 = LogisticRegression(max_iter=1000, class_weight="balanced").fit(X4_train, y4_train)
rf4  = RandomForestClassifier(n_estimators=300, min_samples_leaf=20, class_weight="balanced",
                              random_state=42, n_jobs=-1).fit(X4_train, y4_train)

log4_prob, log4_pred = log4.predict_proba(X4_test)[:, 1], log4.predict(X4_test)
rf4_prob,  rf4_pred  = rf4.predict_proba(X4_test)[:, 1],  rf4.predict(X4_test)

print("test set", len(X4_test), " no-surgery rate", round(y4_test.mean(), 4))

In [ ]:
rq4_results = pd.DataFrame({
    "Logistic regression": [
        cross_val_score(log4, X4_train, y4_train, cv=5, scoring="roc_auc").mean(),
        roc_auc_score(y4_test, log4_prob),
        accuracy_score(y4_test, log4_pred),
        precision_score(y4_test, log4_pred),
        recall_score(y4_test, log4_pred),
        f1_score(y4_test, log4_pred)],
    "Random forest": [
        cross_val_score(rf4, X4_train, y4_train, cv=5, scoring="roc_auc").mean(),
        roc_auc_score(y4_test, rf4_prob),
        accuracy_score(y4_test, rf4_pred),
        precision_score(y4_test, rf4_pred),
        recall_score(y4_test, rf4_pred),
        f1_score(y4_test, rf4_pred)],
}, index=["CV AUROC", "AUROC", "Accuracy", "Precision", "Recall", "F1"]).round(3)

rq4_results

In [ ]:
#Confusion matrix
print(confusion_matrix(y4_test, log4_pred))
print(confusion_matrix(y4_test, rf4_pred))

In [ ]:
#permutation importance
perm4 = permutation_importance(rf4, X4_test, y4_test, n_repeats=5,
                               scoring="roc_auc", random_state=42, n_jobs=-1)

imp4 = pd.DataFrame({
    "RF permutation":  perm4.importances_mean,
    "RF impurity":     rf4.feature_importances_,
    "Logistic |coef|": np.abs(log4.coef_[0]),
}, index=X4_train.columns).sort_values("RF permutation", ascending=False).round(4)

imp4

In [ ]:
#permutation importance figure
imp4["RF permutation"].sort_values().plot(kind="barh", figsize=(8, 8), color="darkorange")

plt.xlabel("Drop in AUROC when the feature is shuffled")
plt.title("RQ4 feature importance")
plt.tight_layout()
plt.show()

In [ ]:
pip install xgboost

In [ ]:
# RQ2 XGBoost
from xgboost import XGBClassifier

X_train_x, X_test_x = X_train.copy(), X_test.copy()
X_train_x.columns = X_train.columns.str.replace("<=", "up to ", regex=False).str.replace("<", "under ", regex=False)
X_test_x.columns  = X_train_x.columns

xgb2 = XGBClassifier(n_estimators=300, max_depth=4, learning_rate=0.1,
                     eval_metric="logloss", random_state=42, n_jobs=-1).fit(X_train_x, y_train)

xgb2_prob, xgb2_pred = xgb2.predict_proba(X_test_x)[:, 1], xgb2.predict(X_test_x)

len(X_train_x), len(X_test_x)

In [ ]:
rq2_results = pd.DataFrame({
    "Logistic regression": [
        cross_val_score(log2, X_train, y_train, cv=5, scoring="roc_auc").mean(),
        roc_auc_score(y_test, log2_prob),
        accuracy_score(y_test, log2_pred),
        precision_score(y_test, log2_pred),
        recall_score(y_test, log2_pred),
        f1_score(y_test, log2_pred)],
    "Random forest": [
        cross_val_score(rf2, X_train, y_train, cv=5, scoring="roc_auc").mean(),
        roc_auc_score(y_test, rf2_prob),
        accuracy_score(y_test, rf2_pred),
        precision_score(y_test, rf2_pred),
        recall_score(y_test, rf2_pred),
        f1_score(y_test, rf2_pred)],
    "XGBoost": [
        cross_val_score(xgb2, X_train_x, y_train, cv=5, scoring="roc_auc").mean(),
        roc_auc_score(y_test, xgb2_prob),
        accuracy_score(y_test, xgb2_pred),
        precision_score(y_test, xgb2_pred),
        recall_score(y_test, xgb2_pred),
        f1_score(y_test, xgb2_pred)],
}, index=["CV AUROC", "AUROC", "Accuracy", "Precision", "Recall", "F1"]).round(3)

rq2_results

In [ ]:
#Confusion matrix
print(confusion_matrix(y_test, xgb2_pred))

In [ ]:
# RQ4 XGBoost
X4_train_x, X4_test_x = X4_train.copy(), X4_test.copy()
X4_train_x.columns = X4_train.columns.str.replace("<=", "up to ", regex=False).str.replace("<", "under ", regex=False)
X4_test_x.columns  = X4_train_x.columns

weight4 = (y4_train == 0).sum() / (y4_train == 1).sum()

xgb4 = XGBClassifier(n_estimators=300, max_depth=4, learning_rate=0.1,
                     scale_pos_weight=weight4, eval_metric="logloss",
                     random_state=42, n_jobs=-1).fit(X4_train_x, y4_train)

xgb4_prob, xgb4_pred = xgb4.predict_proba(X4_test_x)[:, 1], xgb4.predict(X4_test_x)


In [ ]:
rq4_results = pd.DataFrame({
    "Logistic regression": [
        cross_val_score(log4, X4_train, y4_train, cv=5, scoring="roc_auc").mean(),
        roc_auc_score(y4_test, log4_prob),
        accuracy_score(y4_test, log4_pred),
        precision_score(y4_test, log4_pred),
        recall_score(y4_test, log4_pred),
        f1_score(y4_test, log4_pred)],
    "Random forest": [
        cross_val_score(rf4, X4_train, y4_train, cv=5, scoring="roc_auc").mean(),
        roc_auc_score(y4_test, rf4_prob),
        accuracy_score(y4_test, rf4_pred),
        precision_score(y4_test, rf4_pred),
        recall_score(y4_test, rf4_pred),
        f1_score(y4_test, rf4_pred)],
    "XGBoost": [
        cross_val_score(xgb4, X4_train_x, y4_train, cv=5, scoring="roc_auc").mean(),
        roc_auc_score(y4_test, xgb4_prob),
        accuracy_score(y4_test, xgb4_pred),
        precision_score(y4_test, xgb4_pred),
        recall_score(y4_test, xgb4_pred),
        f1_score(y4_test, xgb4_pred)],
}, index=["CV AUROC", "AUROC", "Accuracy", "Precision", "Recall", "F1"]).round(3)

rq4_results

In [ ]:
#Confusion matrix
print(confusion_matrix(y4_test, xgb4_pred))